# Used Car Resale Price — Data Preprocessing

A complete preprocessing workflow on the Used Car Resale dataset: outlier detection and handling (IQR method), categorical encoding (ordinal and nominal), feature/target separation, train-test split, and feature scaling — with scalers fitted only on the training data to avoid data leakage.

## 1. Loading and Inspecting the Dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Day12_Used_Car_Preprocessing_Dataset.csv")
print("Dataset loaded successfully.")
df.head()

Dataset loaded successfully.


,Car_ID,Brand,Year,Mileage_Km,Engine_CC,Power_BHP,Fuel_Type,Transmission,City,Seller_Type,Condition,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
0,CAR0001,Skoda,2021,69708,1152,128.8,Diesel,Manual,Lucknow,Individual,Good,1,0,72,6.38
1,CAR0002,Toyota,2020,88881,903,146.5,Diesel,Automatic,Chandigarh,Individual,Good,1,0,87,4.83
2,CAR0003,Volkswagen,2021,43646,1446,185.9,Diesel,Automatic,Hyderabad,Individual,Very Good,2,0,90,7.30
3,CAR0004,Tata,2019,70847,2069,148.8,Petrol,Manual,Lucknow,Individual,Excellent,3,0,66,3.82
4,CAR0005,Tata,2016,101228,1657,206.0,Petrol,Automatic,Ahmedabad,Dealer,Very Good,2,0,84,1.93


In [2]:
print("Shape (rows, columns):", df.shape)
df.info()

Shape (rows, columns): (320, 15)
<class 'pandas.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Car_ID              320 non-null    str    
 1   Brand               320 non-null    str    
 2   Year                320 non-null    int64  
 3   Mileage_Km          320 non-null    int64  
 4   Engine_CC           320 non-null    int64  
 5   Power_BHP           320 non-null    float64
 6   Fuel_Type           320 non-null    str    
 7   Transmission        320 non-null    str    
 8   City                320 non-null    str    
 9   Seller_Type         320 non-null    str    
 10  Condition           320 non-null    str    
 11  Previous_Owners     320 non-null    int64  
 12  Accidents_Reported  320 non-null    int64  
 13  Service_Score       320 non-null    int64  
 14  Resale_Price_Lakh   320 non-null    float64
dtypes: float64(2), int64(6), str(7)
mem

In [3]:
df.isnull().sum()

Car_ID                0
Brand                 0
Year                  0
Mileage_Km            0
Engine_CC             0
Power_BHP             0
Fuel_Type             0
Transmission          0
City                  0
Seller_Type           0
Condition             0
Previous_Owners       0
Accidents_Reported    0
Service_Score         0
Resale_Price_Lakh     0
dtype: int64

In [4]:
df.describe()

,Year,Mileage_Km,Engine_CC,Power_BHP,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
count,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000
mean,2019.537500,74110.203125,1346.703125,150.489688,1.668750,0.243750,76.203125,4.963031
std,3.341367,38885.260771,543.408160,36.665353,0.865369,0.528164,12.745864,3.359259
min,2014.000000,700.000000,600.000000,51.400000,1.000000,0.000000,55.000000,1.200000
25%,2017.000000,46323.250000,1004.750000,128.450000,1.000000,0.000000,64.750000,2.277500
50%,2020.000000,72718.500000,1303.000000,150.750000,1.000000,0.000000,77.000000,4.610000
75%,2022.000000,97951.500000,1635.250000,171.475000,2.000000,0.000000,87.000000,6.835000
max,2025.000000,320000.000000,5000.000000,390.000000,4.000000,2.000000,98.000000,28.500000


In [5]:
# Checking categorical columns
categorical_cols = ["Brand", "Fuel_Type", "Transmission", "City", "Seller_Type", "Condition"]
for col in categorical_cols:
    print(f"{col}: {sorted(df[col].unique().tolist())}")

Brand: ['Honda', 'Hyundai', 'Kia', 'Mahindra', 'Maruti', 'Renault', 'Skoda', 'Tata', 'Toyota', 'Volkswagen']
Fuel_Type: ['CNG', 'Diesel', 'Electric', 'Petrol']
Transmission: ['Automatic', 'Manual']
City: ['Ahmedabad', 'Bengaluru', 'Chandigarh', 'Delhi', 'Hyderabad', 'Jaipur', 'Kochi', 'Lucknow', 'Mumbai', 'Pune']
Seller_Type: ['Certified Dealer', 'Dealer', 'Individual']
Condition: ['Excellent', 'Fair', 'Good', 'Poor', 'Very Good']


No missing values were found. The dataset has a mix of numeric columns (`Year`, `Mileage_Km`, `Engine_CC`, `Power_BHP`, `Previous_Owners`, `Accidents_Reported`, `Service_Score`) and categorical columns (`Brand`, `Fuel_Type`, `Transmission`, `City`, `Seller_Type`, `Condition`), plus the target `Resale_Price_Lakh`.

## 2. Outlier Detection Using the IQR Method

We use the Interquartile Range (IQR) method: values below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR` are flagged as outliers. This is applied to the continuous numeric features — `Mileage_Km`, `Engine_CC`, and `Power_BHP`. Discrete count-based columns like `Previous_Owners` and `Accidents_Reported` are not treated with IQR, since they have a naturally limited, non-continuous range (e.g. 0–4 owners) where the IQR method tends to over-flag valid values as outliers.

In [6]:
def iqr_bounds(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return lower, upper

continuous_cols = ["Mileage_Km", "Engine_CC", "Power_BHP"]

for col in continuous_cols:
    lower, upper = iqr_bounds(df[col])
    outlier_count = df[(df[col] < lower) | (df[col] > upper)].shape[0]
    print(f"{col}: lower={lower:.2f}, upper={upper:.2f}, outliers={outlier_count}")

Mileage_Km: lower=-31119.12, upper=175393.88, outliers=2
Engine_CC: lower=59.00, upper=2581.00, outliers=6
Power_BHP: lower=63.91, upper=236.01, outliers=7


### Handling the Outliers

Rather than dropping these rows (which would discard otherwise valid cars), we **cap (winsorize)** the outlier values at the IQR boundaries. This keeps every record in the dataset while limiting the influence of extreme values on later scaling and modeling steps.

In [7]:
df_before_capping = df.copy()

for col in continuous_cols:
    lower, upper = iqr_bounds(df[col])
    df[col] = df[col].clip(lower=lower, upper=upper)

# Confirm no more IQR outliers remain in these columns
for col in continuous_cols:
    lower, upper = iqr_bounds(df[col])
    outlier_count = df[(df[col] < lower) | (df[col] > upper)].shape[0]
    print(f"{col} after capping -> remaining outliers: {outlier_count}")

Mileage_Km after capping -> remaining outliers: 0
Engine_CC after capping -> remaining outliers: 0
Power_BHP after capping -> remaining outliers: 0


In [8]:
# Compare summary statistics before and after capping
comparison = pd.DataFrame({
    "Before_Mean": df_before_capping[continuous_cols].mean(),
    "After_Mean": df[continuous_cols].mean(),
    "Before_Max": df_before_capping[continuous_cols].max(),
    "After_Max": df[continuous_cols].max()
})
comparison

,Before_Mean,After_Mean,Before_Max,After_Max
Mileage_Km,74110.203125,73331.414844,320000.0,175393.8750
Engine_CC,1346.703125,1323.546875,5000.0,2581.0000
Power_BHP,150.489688,149.726211,390.0,236.0125


## 3. Encoding Categorical Variables

Two encoding strategies are used, depending on whether a column has a natural order:

- **Ordinal encoding** for `Condition`, since it has a clear order: Poor < Fair < Good < Very Good < Excellent.
- **Label (binary) encoding** for `Transmission`, since it only has two values (Manual/Automatic).
- **One-hot encoding** for `Brand`, `Fuel_Type`, `City`, and `Seller_Type`, since these are nominal categories with no inherent order.

In [9]:
# Ordinal encoding for Condition
condition_order = {"Poor": 1, "Fair": 2, "Good": 3, "Very Good": 4, "Excellent": 5}
df["Condition_Encoded"] = df["Condition"].map(condition_order)

df[["Condition", "Condition_Encoded"]].drop_duplicates().sort_values("Condition_Encoded")

,Condition,Condition_Encoded
29,Poor,1
7,Fair,2
0,Good,3
2,Very Good,4
3,Excellent,5


In [10]:
# Binary/label encoding for Transmission
df["Transmission_Encoded"] = df["Transmission"].map({"Manual": 0, "Automatic": 1})

df[["Transmission", "Transmission_Encoded"]].drop_duplicates()

,Transmission,Transmission_Encoded
0,Manual,0
1,Automatic,1


In [11]:
# One-hot encoding for nominal columns: Brand, Fuel_Type, City, Seller_Type
nominal_cols = ["Brand", "Fuel_Type", "City", "Seller_Type"]
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

print("Shape after one-hot encoding:", df.shape)
df.head()

Shape after one-hot encoding: (320, 36)


,Car_ID,Year,Mileage_Km,Engine_CC,Power_BHP,Transmission,Condition,Previous_Owners,Accidents_Reported,Service_Score,...,City_Chandigarh,City_Delhi,City_Hyderabad,City_Jaipur,City_Kochi,City_Lucknow,City_Mumbai,City_Pune,Seller_Type_Dealer,Seller_Type_Individual
0,CAR0001,2021,69708.0,1152,128.8,Manual,Good,1,0,72,...,False,False,False,False,False,True,False,False,False,True
1,CAR0002,2020,88881.0,903,146.5,Automatic,Good,1,0,87,...,True,False,False,False,False,False,False,False,False,True
2,CAR0003,2021,43646.0,1446,185.9,Automatic,Very Good,2,0,90,...,False,False,True,False,False,False,False,False,False,True
3,CAR0004,2019,70847.0,2069,148.8,Manual,Excellent,3,0,66,...,False,False,False,False,False,True,False,False,False,True
4,CAR0005,2016,101228.0,1657,206.0,Automatic,Very Good,2,0,84,...,False,False,False,False,False,False,False,False,True,False


In [12]:
# Drop the original text columns now replaced by encoded versions
df = df.drop(columns=["Condition", "Transmission"])
df.head()

,Car_ID,Year,Mileage_Km,Engine_CC,Power_BHP,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh,Condition_Encoded,...,City_Chandigarh,City_Delhi,City_Hyderabad,City_Jaipur,City_Kochi,City_Lucknow,City_Mumbai,City_Pune,Seller_Type_Dealer,Seller_Type_Individual
0,CAR0001,2021,69708.0,1152,128.8,1,0,72,6.38,3,...,False,False,False,False,False,True,False,False,False,True
1,CAR0002,2020,88881.0,903,146.5,1,0,87,4.83,3,...,True,False,False,False,False,False,False,False,False,True
2,CAR0003,2021,43646.0,1446,185.9,2,0,90,7.30,4,...,False,False,True,False,False,False,False,False,False,True
3,CAR0004,2019,70847.0,2069,148.8,3,0,66,3.82,5,...,False,False,False,False,False,True,False,False,False,True
4,CAR0005,2016,101228.0,1657,206.0,2,0,84,1.93,4,...,False,False,False,False,False,False,False,False,True,False


## 4. Separating Features and Target Variable

In [13]:
# Car_ID is just an identifier, not a predictive feature
X = df.drop(columns=["Car_ID", "Resale_Price_Lakh"])
y = df["Resale_Price_Lakh"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)
X.columns.tolist()

Features shape:

 (320, 32)
Target shape: (320,)


['Year',
 'Mileage_Km',
 'Engine_CC',
 'Power_BHP',
 'Previous_Owners',
 'Accidents_Reported',
 'Service_Score',
 'Condition_Encoded',
 'Transmission_Encoded',
 'Brand_Hyundai',
 'Brand_Kia',
 'Brand_Mahindra',
 'Brand_Maruti',
 'Brand_Renault',
 'Brand_Skoda',
 'Brand_Tata',
 'Brand_Toyota',
 'Brand_Volkswagen',
 'Fuel_Type_Diesel',
 'Fuel_Type_Electric',
 'Fuel_Type_Petrol',
 'City_Bengaluru',
 'City_Chandigarh',
 'City_Delhi',
 'City_Hyderabad',
 'City_Jaipur',
 'City_Kochi',
 'City_Lucknow',
 'City_Mumbai',
 'City_Pune',
 'Seller_Type_Dealer',
 'Seller_Type_Individual']

## 5. Train-Test Split

Splitting into training (80%) and testing (20%) sets **before** scaling, so that scaling parameters are learned only from the training data.

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (256, 32)
X_test shape: (64, 32)
y_train shape: (256,)
y_test shape: (64,)


## 6. Feature Scaling (Avoiding Data Leakage)

Numeric features are standardized using `StandardScaler` (zero mean, unit variance), since the features have very different ranges (e.g. `Mileage_Km` in the tens of thousands vs. `Previous_Owners` from 0–4).

**Important:** the scaler is `fit()` only on `X_train`, and then used to `transform()` both `X_train` and `X_test`. This ensures no information from the test set leaks into the scaling parameters (mean/standard deviation).

In [15]:
from sklearn.preprocessing import StandardScaler

numeric_features_to_scale = ["Year", "Mileage_Km", "Engine_CC", "Power_BHP",
                              "Previous_Owners", "Accidents_Reported", "Service_Score"]

scaler = StandardScaler()

# Fit ONLY on training data
scaler.fit(X_train[numeric_features_to_scale])

# Transform both training and test data using the same fitted scaler
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_features_to_scale] = scaler.transform(X_train[numeric_features_to_scale])
X_test_scaled[numeric_features_to_scale] = scaler.transform(X_test[numeric_features_to_scale])

X_train_scaled[numeric_features_to_scale].describe()

,Year,Mileage_Km,Engine_CC,Power_BHP,Previous_Owners,Accidents_Reported,Service_Score
count,2.560000e+02,2.560000e+02,2.560000e+02,2.560000e+02,2.560000e+02,2.560000e+02,2.560000e+02
mean,-3.816392e-17,-1.040834e-17,1.214306e-17,-1.734723e-16,-6.938894e-18,3.816392e-17,-6.938894e-18
std,1.001959e+00,1.001959e+00,1.001959e+00,1.001959e+00,1.001959e+00,1.001959e+00,1.001959e+00
min,-1.698227e+00,-2.046627e+00,-1.632432e+00,-2.697210e+00,-7.343793e-01,-4.426342e-01,-1.738645e+00
25%,-7.893502e-01,-7.699804e-01,-7.120323e-01,-6.394782e-01,-7.343793e-01,-4.426342e-01,-8.755947e-01
50%,1.195268e-01,-1.044397e-02,-5.555874e-02,1.214335e-02,-7.343793e-01,-4.426342e-01,1.078813e-01
75%,7.254448e-01,6.887173e-01,6.944581e-01,6.582891e-01,4.333294e-01,-4.426342e-01,7.501513e-01
max,1.634322e+00,2.884383e+00,2.779694e+00,2.687859e+00,2.768747e+00,3.398530e+00,1.713556e+00


In [16]:
# Verify: training data should have ~0 mean and ~1 std after scaling
print("Training data (scaled) mean:\n", X_train_scaled[numeric_features_to_scale].mean().round(3))
print("\nTraining data (scaled) std:\n", X_train_scaled[numeric_features_to_scale].std().round(3))

# Test data won't be exactly 0/1 since it was transformed using the TRAINING set's parameters -- this is expected and correct
print("\nTest data (scaled) mean:\n", X_test_scaled[numeric_features_to_scale].mean().round(3))

Training data (scaled) mean:
 Year                 -0.0
Mileage_Km           -0.0
Engine_CC             0.0
Power_BHP            -0.0
Previous_Owners      -0.0
Accidents_Reported    0.0
Service_Score        -0.0
dtype: float64

Training data (scaled) std:
 Year                  1.002
Mileage_Km            1.002
Engine_CC             1.002
Power_BHP             1.002
Previous_Owners       1.002
Accidents_Reported    1.002
Service_Score         1.002
dtype: float64

Test data (scaled) mean:
 Year                 -0.103
Mileage_Km            0.001
Engine_CC            -0.105
Power_BHP            -0.060
Previous_Owners       0.233
Accidents_Reported    0.128
Service_Score        -0.182
dtype: float64


## 7. Verifying the Preprocessed Dataset

In [17]:
print("Final X_train shape:", X_train_scaled.shape)
print("Final X_test shape:", X_test_scaled.shape)
print("\nAny missing values in X_train?", X_train_scaled.isnull().sum().sum())
print("Any missing values in X_test?", X_test_scaled.isnull().sum().sum())
print("\nData types in X_train:")
X_train_scaled.dtypes

Final X_train shape: (256, 32)
Final X_test shape: (64, 32)

Any missing values in X_train? 0
Any missing values in X_test? 0

Data types in X_train:


Year                      float64
Mileage_Km                float64
Engine_CC                 float64
Power_BHP                 float64
Previous_Owners           float64
Accidents_Reported        float64
Service_Score             float64
Condition_Encoded           int64
Transmission_Encoded        int64
Brand_Hyundai                bool
Brand_Kia                    bool
Brand_Mahindra               bool
Brand_Maruti                 bool
Brand_Renault                bool
Brand_Skoda                  bool
Brand_Tata                   bool
Brand_Toyota                 bool
Brand_Volkswagen             bool
Fuel_Type_Diesel             bool
Fuel_Type_Electric           bool
Fuel_Type_Petrol             bool
City_Bengaluru               bool
City_Chandigarh              bool
City_Delhi                   bool
City_Hyderabad               bool
City_Jaipur                  bool
City_Kochi                   bool
City_Lucknow                 bool
City_Mumbai                  bool
City_Pune     

In [18]:
X_train_scaled.head()

,Year,Mileage_Km,Engine_CC,Power_BHP,Previous_Owners,Accidents_Reported,Service_Score,Condition_Encoded,Transmission_Encoded,Brand_Hyundai,...,City_Chandigarh,City_Delhi,City_Hyderabad,City_Jaipur,City_Kochi,City_Lucknow,City_Mumbai,City_Pune,Seller_Type_Dealer,Seller_Type_Individual
132,-0.486391,-0.225904,-0.322825,0.303144,-0.734379,-0.442634,-0.454105,4,0,False,...,False,False,False,False,False,True,False,False,False,False
317,0.725445,0.089296,-0.656909,-0.328921,-0.734379,1.477948,-0.614672,3,1,False,...,False,True,False,False,False,False,False,False,True,False
234,-1.395268,1.115547,-0.120148,0.418918,-0.734379,-0.442634,-0.534389,3,0,False,...,False,False,False,False,True,False,False,False,False,True
312,1.028404,-0.195102,0.481202,0.412660,0.433329,-0.442634,0.188165,3,0,False,...,False,False,False,False,False,False,False,True,True,False
232,-1.092309,0.706499,0.788558,-0.435309,0.433329,-0.442634,0.107881,4,1,False,...,False,True,False,False,False,False,False,False,False,True


## 8. Exporting the Preprocessed Data

In [19]:
# Combine features and target for saving, keeping train and test separate
train_processed = X_train_scaled.copy()
train_processed["Resale_Price_Lakh"] = y_train.values

test_processed = X_test_scaled.copy()
test_processed["Resale_Price_Lakh"] = y_test.values

train_processed.to_csv("Used_Car_Preprocessed_Train.csv", index=False)
test_processed.to_csv("Used_Car_Preprocessed_Test.csv", index=False)

print("Exported 'Used_Car_Preprocessed_Train.csv' and 'Used_Car_Preprocessed_Test.csv'")

Exported 'Used_Car_Preprocessed_Train.csv' and 'Used_Car_Preprocessed_Test.csv'


## 9. Summary of Preprocessing Decisions

1. **Outlier handling:** Used the IQR method on the continuous features `Mileage_Km`, `Engine_CC`, and `Power_BHP`. Rather than dropping outlier rows (which would lose valid car records), values were **capped (winsorized)** at the IQR boundaries. Discrete count columns (`Previous_Owners`, `Accidents_Reported`) were left untouched, since IQR isn't well-suited to their naturally narrow, discrete range.

2. **Categorical encoding:**
   - `Condition` → **ordinal encoding** (Poor=1 → Excellent=5), since it has a clear natural order.
   - `Transmission` → **binary/label encoding** (Manual=0, Automatic=1), since it's a two-category column.
   - `Brand`, `Fuel_Type`, `City`, `Seller_Type` → **one-hot encoding**, since these are nominal categories with no inherent ranking.

3. **Feature/target separation:** `Car_ID` was dropped as a non-predictive identifier; `Resale_Price_Lakh` was set aside as the target variable `y`, with all remaining columns forming the feature matrix `X`.

4. **Train-test split:** Performed an 80/20 split **before** any scaling, using `random_state=42` for reproducibility.

5. **Feature scaling and leakage prevention:** Applied `StandardScaler` to the numeric features. The scaler was **fit only on `X_train`**, then used to transform both `X_train` and `X_test` — ensuring the test set's statistics never influenced the scaling parameters, which avoids data leakage.

6. **Verification:** Confirmed the final processed train/test sets have no missing values, consistent shapes, and correct data types before export.